<a href="https://colab.research.google.com/github/emiliano-ipa/AST1-CEIA-UBA-TP2/blob/main/Desafios/Desafio_1/Desafio_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Procesamiento de Lenguaje Natural I
## Desafío 1: Vectorización de texto y clasificación con Naïve Bayes

**Alumno:** Emiliano Iparraguirre
**Correo:** emiliano.iparraguirre22@gmail.com
**Código de alumno:** a2010

Especialización en Inteligencia Artificial (CEIA), FIUBA

### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [1]:
%pip install numpy scikit-learn

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [10]:
# tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [11]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [12]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [13]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [14]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [15]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [16]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [17]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 9019, 9016, 8748], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [18]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [19]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [20]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [21]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [22]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [23]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## Resolución del Desafío 1

### Ejercicio 1: Similaridad de documentos

**Consigna:** Tomar 5 documentos al azar y medir su similaridad con el resto de los documentos. Estudiar los 5 documentos más similares de cada uno y analizar si la similaridad tiene sentido según el contenido del texto y la etiqueta de clasificación.

**Enfoque:**
- Representamos cada documento de train con TF-IDF, usando el vectorizador por defecto como línea base.
- Para cada documento elegido calculamos la similitud coseno contra todo el corpus de train y nos quedamos con los 5 más cercanos, salteando el propio documento.
- Para justificar cada similitud no solo miramos la etiqueta. También identificamos qué palabras generan el coseno y si el vecino pertenece a una clase "hermana", es decir, de la misma familia temática.

In [24]:
# 1.1 – Vectorización y selección de documentos
import random
import pandas as pd

pd.set_option('display.max_colwidth', None)   # para ver las columnas de texto completas

# Vectorizador TF-IDF por defecto, ajustado solo con train
tfidfvect = TfidfVectorizer()
X_train = tfidfvect.fit_transform(newsgroups_train.data)
y_train = newsgroups_train.target
target_names = newsgroups_train.target_names
idx2word = {v: k for k, v in tfidfvect.vocabulary_.items()}

# Al quitar headers, footers y quotes, algunos documentos quedan vacíos.
# Su vector es todo ceros y su coseno con cualquier otro es 0, así que conviene saberlo.
n_terminos = X_train.getnnz(axis=1)
print(f'Documentos vacíos en train: {(n_terminos == 0).sum()} de {X_train.shape[0]}')

# Selección aleatoria de 5 documentos, con semilla para que sea reproducible
random.seed(42)
idx_azar = random.sample(range(len(newsgroups_train.data)), 5)
print(f'Documentos elegidos al azar: {idx_azar}')
print(f'Términos distintos de cada uno: {[int(n_terminos[i]) for i in idx_azar]}')

Documentos vacíos en train: 311 de 11314
Documentos elegidos al azar: [10476, 1824, 409, 4506, 4012]
Términos distintos de cada uno: [82, 106, 39, 32, 28]


#### Interpretación: preparación de los datos

**Representación.** Ajustamos un `TfidfVectorizer` por defecto únicamente con train, que funciona como línea base. Cada documento queda representado como un vector de 101.631 dimensiones, una por término del vocabulario, y casi todas sus componentes son cero. Como sklearn normaliza cada vector con norma L2, la similitud coseno entre dos documentos equivale al producto punto de sus vectores, es decir, a la suma término a término de sus pesos TF-IDF. Esto permite descomponer cada similitud en el aporte de cada palabra, y lo usamos más adelante para justificar los resultados.

**Documentos vacíos.** 311 de los 11.314 documentos de train (2,7%) quedan sin ningún término al quitar encabezados, firmas y citas (`remove=('headers', 'footers', 'quotes')`). Son mayormente respuestas que solo citaban otro mensaje. Su vector es nulo, de modo que su similitud coseno con cualquier documento es 0 y no pueden compararse. Ninguno de los 5 documentos elegidos está vacío, así que en este ejercicio no afectan. Sí los vamos a considerar en el Ejercicio 2: si un documento de test queda vacío, `argmax` le asigna por defecto el primer documento de train y su clase, lo que introduce un error sistemático.

**Selección y reproducibilidad.** Con `random.seed(42)` se eligen los documentos `[10476, 1824, 409, 4506, 4012]`. La semilla garantiza que el análisis sea reproducible.

**Largo de los documentos.** Los documentos elegidos tienen entre 28 y 106 términos distintos:

| Doc | Clase | Términos distintos |
|---|---|---|
| 10476 | rec.sport.hockey | 82 |
| 1824 | comp.sys.mac.hardware | 106 |
| 409 | comp.graphics | 39 |
| 4506 | rec.autos | 32 |
| 4012 | rec.sport.hockey | 28 |

**Hipótesis a verificar.** En un documento con pocos términos, la similitud depende de muy pocas palabras. Si alguna de ellas es ambigua, como un nombre propio o una ciudad, puede atraer vecinos de otro tema. Sin embargo, esperamos que importe menos la cantidad de términos que lo distintivos que sean. Un documento corto con una palabra muy específica, como el nombre de un software, puede encontrar vecinos coherentes. Un documento más largo, en cambio, puede encontrar vecinos sin relación si lo que comparte con ellos son palabras genéricas, ya que el vectorizador por defecto no elimina stop words. Contrastamos esta hipótesis en la siguiente sección, al analizar qué términos explican cada similitud.

In [26]:
# 1.2 – Funciones de análisis
# Familias temáticas: agrupan clases "hermanas" que comparten vocabulario
def familia(clase):
    nombre = target_names[clase]
    if nombre in ('alt.atheism', 'soc.religion.christian', 'talk.religion.misc'):
        return 'religión'
    if nombre.startswith('talk.politics'):
        return 'política'
    if nombre.startswith('comp.'):
        return 'computación'
    if nombre.startswith('rec.sport'):
        return 'deportes'
    if nombre in ('rec.autos', 'rec.motorcycles'):
        return 'vehículos'
    if nombre.startswith('sci.'):
        return 'ciencia'
    return 'misc'

def resumen(texto, n=250):
    """Texto en una sola línea, recortado a n caracteres."""
    return ' '.join(texto.split())[:n]

def terminos_que_explican(i, j, X=X_train, vocab=idx2word, n=6):
    """
    Términos que más aportan a la similitud coseno entre los documentos i y j.
    Como los vectores están normalizados con L2, el coseno es la suma de
    tfidf_i * tfidf_j término a término. Cada aporte se expresa como % del coseno.
    """
    contrib = X[i].multiply(X[j]).tocoo()
    total = contrib.data.sum()
    orden = np.argsort(contrib.data)[::-1][:n]
    return ', '.join(f"{vocab[contrib.col[k]]} ({contrib.data[k] / total:.0%})" for k in orden)

In [28]:
# 1.3 – Los 5 más similares de cada documento
resumenes = []

for idx in idx_azar:
    cossim = cosine_similarity(X_train[idx], X_train)[0]
    mostsim = np.argsort(cossim)[::-1][1:6]   # salteamos el propio documento (coseno = 1)
    clase_orig = y_train[idx]

    print('=' * 110)
    print(f'DOCUMENTO {idx} | clase: {target_names[clase_orig]} | términos distintos: {n_terminos[idx]}')
    print(f'Texto: {resumen(newsgroups_train.data[idx], 400)}')

    tabla = pd.DataFrame({
        'doc': mostsim,
        'coseno': cossim[mostsim].round(3),
        'clase': [target_names[y_train[j]] for j in mostsim],
        'misma clase': ['sí' if y_train[j] == clase_orig else 'no' for j in mostsim],
        'misma familia': ['sí' if familia(y_train[j]) == familia(clase_orig) else 'no' for j in mostsim],
        'términos que explican la similitud': [terminos_que_explican(idx, j) for j in mostsim],
        'texto': [resumen(newsgroups_train.data[j], 150) for j in mostsim],
    })
    display(tabla)

    resumenes.append({
        'doc': idx,
        'clase': target_names[clase_orig],
        'términos': n_terminos[idx],
        'coseno medio top 5': cossim[mostsim].mean().round(3),
        'misma clase (de 5)': (y_train[mostsim] == clase_orig).sum(),
        'misma familia (de 5)': sum(familia(y_train[j]) == familia(clase_orig) for j in mostsim),
    })

print('=' * 110)
print('RESUMEN DE LOS 5 DOCUMENTOS')
display(pd.DataFrame(resumenes))

DOCUMENTO 10476 | clase: rec.sport.hockey | términos distintos: 82
Texto: This is a general question for US readers: How extensive is the playoff coverage down there? In Canada, it is almost impossible not to watch a series on TV (ie the only two series I have not had an opportunity to watch this year are Wash-NYI and Chi-Stl, the latter because I'm in the wrong time zone!). We (in Canada) are basically swamped with coverage, and I wonder how many series/games are telev


,doc,coseno,clase,misma clase,misma familia,términos que explican la similitud,texto
0,5064,0.225,rec.sport.hockey,sí,sí,"games (26%), series (22%), watch (16%), playoff (12%), year (4%), this (4%)","I only have one comment on this: You call this a *classic* playoff year and yet you don't include a Chicago-Detroit series. C'mon, I'm a Boston fan an"
1,9623,0.217,talk.politics.mideast,no,no,"the (19%), and (9%), in (7%), to (6%), us (6%), they (4%)",Accounts of Anti-Armenian Human Right Violations in Azerbaijan #012 Prelude to Current Events in Nagorno-Karabakh +-----------------------------------
2,10575,0.216,sci.crypt,no,no,"the (18%), to (9%), not (8%), is (8%), in (5%), and (5%)","I am not an expert in the cryptography science, but some basic things seem evident to me, things which this Clinton Clipper do not address. The all pe"
3,10836,0.213,alt.atheism,no,no,"the (23%), is (13%), in (9%), to (6%), not (6%), and (4%)",Archive-name: atheism/faq Alt-atheism-archive-name: faq Last-modified: 5 April 1993 Version: 1.1 Alt.Atheism Frequently-Asked Questions This file cont
4,2350,0.211,sci.crypt,no,no,"the (21%), is (10%), are (8%), in (8%), and (8%), to (6%)","Archive-name: net-privacy/part1 Last-modified: 1993/3/3 Version: 2.1 IDENTITY, PRIVACY, and ANONYMITY on the INTERNET ================================"


DOCUMENTO 1824 | clase: comp.sys.mac.hardware | términos distintos: 106
Texto: I think this kind of comparison is pretty useless in general. The processor is only good when a good computer is designed around it adn the computer is used in its designed purpose. Comparing processor speed is pretty dumb because all you have to do is just increase the clock speed to increase speed among other things. I mean how can you say a 040 is faster than a 486 without giving is operational


,doc,coseno,clase,misma clase,misma familia,términos que explican la similitud,texto
0,9921,0.354,comp.sys.mac.hardware,sí,sí,"processor (34%), speed (30%), 040 (7%), clock (4%), is (3%), computer (3%)","dhk@ubbpc.uucp (Dave Kitabjian) writes ... 040 486 030 386 020 286 060 fastest, then Pentium, with the first versions of the PowerPC somewhere in the"
1,6364,0.313,comp.sys.mac.hardware,sí,sí,"processor (48%), 040 (23%), its (6%), to (3%), the (3%), does (2%)","Well folks, after some thought the answer struck me flat in the face: ""Why would Apple release a Duo Dock with a processor of its own?"" Here's why- Pe"
2,5509,0.304,comp.sys.mac.hardware,sí,sí,"processor (37%), speed (18%), 040 (12%), 486 (5%), faster (4%), is (4%)","rvenkate@ux4.cso.uiuc.edu (Ravikuma Venkateswar) writes ... Benchmarks are for marketing dweebs and CPU envy. OK, if it will make you happy, the 486 i"
3,2641,0.250,comp.sys.mac.hardware,sí,sí,"040 (36%), processor (14%), fpu (8%), is (7%), the (5%), you (5%)",I think this is mostly the fault of the people who write up the literature and price lists being confused themselves. Since there are two possible pro
4,4359,0.242,comp.sys.mac.hardware,sí,sí,"processor (42%), fpu (21%), 040 (9%), the (3%), you (3%), is (3%)","If you get the Centris 650 with CD configuration, you are getting a Mac with a 68RC040 processor that has built-in math coprocessor support. My unders"


DOCUMENTO 409 | clase: comp.graphics | términos distintos: 39
Texto: I can't fiqure this out. I have properly compiled pov on a unix machine running SunOS 4.1.3 The problem is that when I run the sample .pov files and use the EXACT same parameters when compiling different .tga outputs. Some of the .tga's are okay, and other's are unrecognizable by any software.


,doc,coseno,clase,misma clase,misma familia,términos que explican la similitud,texto
0,3444,0.231,comp.graphics,sí,sí,"pov (55%), outputs (28%), files (11%), any (3%), and (2%), that (1%)","Hi, I'm just getting into PoVRay and I was wondering if there is a graphic package that outputs .POV files. Any help would be appreciated. Thanks. Lat"
1,5799,0.209,comp.graphics,sí,sí,"tga (68%), files (12%), the (6%), when (4%), same (3%), and (2%)",I finally got a 24 bit viewer for my POVRAY generated .TGA files. It was written in C by Sean Malloy and he kindly sent me a copy. He wrote it for the
2,5905,0.198,comp.graphics,sí,sí,"pov (93%), the (2%), can (2%), and (2%), of (1%)",Hallo POV-Renderers ! I've got a BocaX3 Card. Now I try to get POV displaying True Colors while rendering. I've tried most of the options and UNIVESA-
3,1764,0.184,comp.graphics,sí,sí,"pov (77%), files (5%), are (4%), use (2%), of (2%), and (2%)",hi guys like all people in this group i'm a fans of fractal and render sw my favourite are fractint pov & 3dstudio 2.0 now listen my ideas i'have just
4,3364,0.166,comp.graphics,sí,sí,"tga (61%), files (21%), the (6%), same (5%), any (3%), this (2%)",Does anyone know of a good way (standard PC application/PD utility) to convert tif/img/tga files into LaserJet III format. We would also like to do th


... (2 outputs left out)

DOCUMENTO 4012 | clase: rec.sport.hockey | términos distintos: 28
Texto: For those Leaf fans who are concerned, the following players are slated for return on Thursday's Winnipeg-Toronto game : Peter Zezel, John Cullen Mark Osborne and Dave Ellett are questionable to return on Thursday.


,doc,coseno,clase,misma clase,misma familia,términos que explican la similitud,texto
0,6599,0.160,soc.religion.christian,no,no,"peter (55%), are (20%), those (9%), who (6%), on (6%), the (2%)",True. Also read 2 Peter 3:16 Peter warns that the scriptures are often hard to understand by those who are not learned on the subject.
1,10644,0.143,rec.sport.hockey,sí,sí,"leaf (48%), fans (43%), are (5%), the (2%), to (1%), and (1%)","In <1qvos8$r78@cl.msu.>, vergolin@euler.lbs.msu.edu (David Vergolini) writes... There's quite a few Wings fans lurking about here, they just tend to b"
2,7478,0.136,rec.sport.hockey,sí,sí,"winnipeg (29%), cullen (27%), toronto (23%), ellett (21%)","Toronto 1 1 1--3 Detroit 1 4 1--6 First period 1, Detroit, Yzerman 1 (Gallant, Ciccarelli) 4:48. 2, Toronto, Cullen 1 (Clark, Gill) 10:44. Second peri"
3,7308,0.132,rec.sport.hockey,sí,sí,"toronto (65%), game (16%), players (11%), to (3%), for (2%), the (2%)","Detroit is a very disciplined team. There's a lot of Europeans in Detroit which would make the game fast, so Toronto would have to slow the game down,"
4,10792,0.131,rec.sport.baseball,no,sí,"toronto (57%), fans (21%), are (14%), the (4%), to (3%), and (1%)","The tribe will be in town from April 16 to the 19th. There are ALWAYS tickets available! (Though they are playing Toronto, and many Toronto fans make"


RESUMEN DE LOS 5 DOCUMENTOS


,doc,clase,términos,coseno medio top 5,misma clase (de 5),misma familia (de 5)
0,10476,rec.sport.hockey,82,0.217,1,1
1,1824,comp.sys.mac.hardware,106,0.293,5,5
2,409,comp.graphics,39,0.197,5,5
3,4506,rec.autos,32,0.165,3,4
4,4012,rec.sport.hockey,28,0.140,3,4


#### Interpretación: similaridad de los 5 documentos

| Doc | Clase | Términos | Coseno medio | Misma clase | Misma familia |
|---|---|---|---|---|---|
| 10476 | rec.sport.hockey | 82 | 0,217 | 1/5 | 1/5 |
| 1824 | comp.sys.mac.hardware | 106 | 0,293 | 5/5 | 5/5 |
| 409 | comp.graphics | 39 | 0,197 | 5/5 | 5/5 |
| 4506 | rec.autos | 32 | 0,165 | 3/5 | 4/5 |
| 4012 | rec.sport.hockey | 28 | 0,140 | 3/5 | 4/5 |

**Cuándo funciona.** La similitud por TF-IDF acierta cuando el documento contiene términos distintivos, es decir, frecuentes en ese documento y raros en el resto del corpus. En el doc 1824, *processor*, *speed* y *040* explican la mayor parte de cada similitud y los 5 vecinos son de la misma clase. El doc 409 es aún más claro: aunque tiene solo 39 términos, acierta 5 de 5 porque *pov* y *tga* son términos casi exclusivos de gráficos por computadora, y en un caso *pov* explica el 93% del coseno. Lo mismo ocurre con el resumen de partido vecino del doc 4012, que se conecta únicamente por nombres propios (*winnipeg*, *cullen*, *toronto*, *ellett*). Esto confirma la hipótesis planteada: importa menos la cantidad de términos que lo distintivos que sean.

**Cuándo falla.** Los errores se explican por tres deficiencias del método:

1. **Peso de las palabras funcionales.** En el doc 10476, 4 de los 5 vecinos son de clases sin relación (Medio Oriente, criptografía y ateísmo), y su similitud se explica casi por completo por stop words (*the*, *and*, *in*, *to*, *is*). Son documentos largos (FAQs y archivos de varias partes), que contienen todas las palabras funcionales y por eso se parecen débilmente a cualquier texto. El único vecino correcto (*games*, *series*, *watch*, *playoff*) supera a los incorrectos por muy poco (0,225 contra 0,217). El vocabulario temático del documento es escaso y queda tapado por el ruido gramatical.
2. **Ambigüedad léxica.** La bolsa de palabras no distingue significados. En el doc 4012, *peter* (el jugador Peter Zezel) explica el 55% de la similitud con un texto religioso que cita la segunda epístola de Pedro. En el doc 4506, *sound* explica el 53% de la similitud con un documento de Mac: en el original es un verbo ("this does sound good") y en el vecino es el sonido de la computadora.
3. **Artefactos de preprocesamiento.** La conversión a minúsculas confunde *US* (Estados Unidos) con el pronombre *us*, que aporta un 6% de la similitud con el texto sobre Armenia. Además, el tokenizador por defecto separa "I've" y retiene *ve* como término, que aporta un 8% en el vecino de motos del doc 4506.

**No todos los errores son iguales.** Algunos vecinos de otra clase comparten tema con el documento: la moto que consume aceite (*oil*) frente a la pregunta de autos sobre cambio de aceite, o el partido de béisbol en Toronto frente al de hockey en Toronto. Son clases hermanas y el error es razonable. Otros, como el texto religioso o el de sonido de Mac, no tienen relación temática y surgen solo por coincidencia de palabras.

**El valor del coseno no indica la calidad del vecino.** El doc 409 acierta todos sus vecinos con cosenos de 0,17 a 0,23, más bajos que los del doc 10476, que falla en 4 de 5. Los documentos cortos producen cosenos bajos por construcción, porque comparten pocos términos con cualquier otro. Por eso lo informativo es el orden relativo de los vecinos, no el valor absoluto.

**Conclusión.** TF-IDF con similitud coseno es una representación simple que captura bien la coincidencia temática cuando existe vocabulario específico. Pero mide coincidencia de palabras, no de significado: no resuelve homónimos ni sinónimos, y con la configuración por defecto se deja arrastrar por palabras funcionales y decisiones de preprocesamiento. Las dos primeras limitaciones se pueden mitigar ajustando el vectorizador, por ejemplo eliminando stop words. La ambigüedad de significado es propia de la bolsa de palabras y requiere representaciones que incorporen contexto, como los embeddings.

#### Experimento: eliminación de stop words

**Motivación.** El análisis anterior mostró que, con el vectorizador por defecto, parte de la similitud entre documentos se explica por palabras funcionales (*the*, *and*, *in*, *to*, *is*). Son términos que aparecen en casi cualquier texto en inglés y no aportan información temática. El caso más claro es el doc 10476, donde 4 de los 5 vecinos salieron de clases no relacionadas por este motivo.

**Hipótesis.** Si eliminamos las stop words, la similitud pasará a depender solo de términos con contenido. Esperamos que:
- el doc 10476 encuentre vecinos de hockey o al menos de deportes;
- los documentos que ya funcionaban bien (1824 y 409) mantengan sus vecinos, porque su similitud se explicaba por términos distintivos;
- los errores por ambigüedad léxica (*peter*, *sound*) **no** se corrijan, porque esas palabras no son stop words y el problema es de significado, no de frecuencia.

**Diseño.** Ajustamos un nuevo `TfidfVectorizer(stop_words='english')` sobre train, manteniendo el resto de los parámetros por defecto, y repetimos el análisis sobre los mismos 5 documentos. Para cada uno mostramos los nuevos vecinos, si ya figuraban con el vectorizador anterior y qué términos explican ahora la similitud. Al final comparamos ambos vectorizadores en una tabla.

In [29]:
# 1.4 – Repetición del análisis sin stop words
# Vectorizador con eliminación de stop words (mismo resto de parámetros por defecto)
tfidf_sw = TfidfVectorizer(stop_words='english')
X_sw = tfidf_sw.fit_transform(newsgroups_train.data)
idx2word_sw = {v: k for k, v in tfidf_sw.vocabulary_.items()}
n_terminos_sw = X_sw.getnnz(axis=1)

print(f'Vocabulario: {X_train.shape[1]} términos sin filtrar -> {X_sw.shape[1]} con stop words eliminadas')
print(f'Documentos vacíos: {(n_terminos == 0).sum()} -> {(n_terminos_sw == 0).sum()}')

def vecinos(X, idx, k=5):
    """Índices y cosenos de los k documentos más similares a idx (sin incluirlo)."""
    cossim = cosine_similarity(X[idx], X)[0]
    top = np.argsort(cossim)[::-1][1:k + 1]
    return top, cossim[top]

comparacion = []

for idx in idx_azar:
    clase_orig = y_train[idx]
    top_base, cos_base = vecinos(X_train, idx)
    top_sw, cos_sw = vecinos(X_sw, idx)

    print('=' * 110)
    print(f'DOCUMENTO {idx} | clase: {target_names[clase_orig]} | '
          f'términos: {n_terminos[idx]} -> {n_terminos_sw[idx]} sin stop words')

    tabla = pd.DataFrame({
        'doc': top_sw,
        'coseno': cos_sw.round(3),
        'clase': [target_names[y_train[j]] for j in top_sw],
        'misma clase': ['sí' if y_train[j] == clase_orig else 'no' for j in top_sw],
        'misma familia': ['sí' if familia(y_train[j]) == familia(clase_orig) else 'no' for j in top_sw],
        'ya estaba antes': ['sí' if j in top_base else 'no' for j in top_sw],
        'términos que explican la similitud': [terminos_que_explican(idx, j, X=X_sw, vocab=idx2word_sw) for j in top_sw],
        'texto': [resumen(newsgroups_train.data[j], 150) for j in top_sw],
    })
    print(tabla.to_string())

    comparacion.append({
        'doc': idx,
        'clase': target_names[clase_orig],
        'misma clase (antes)': (y_train[top_base] == clase_orig).sum(),
        'misma clase (después)': (y_train[top_sw] == clase_orig).sum(),
        'misma familia (antes)': sum(familia(y_train[j]) == familia(clase_orig) for j in top_base),
        'misma familia (después)': sum(familia(y_train[j]) == familia(clase_orig) for j in top_sw),
        'coseno medio (antes)': cos_base.mean().round(3),
        'coseno medio (después)': cos_sw.mean().round(3),
        'vecinos que se mantienen': len(set(top_base) & set(top_sw)),
    })

print('=' * 110)
print('COMPARACIÓN: VECTORIZADOR POR DEFECTO vs. SIN STOP WORDS')
print(pd.DataFrame(comparacion).to_string())

Vocabulario: 101631 términos sin filtrar -> 101322 con stop words eliminadas
Documentos vacíos: 311 -> 318
DOCUMENTO 10476 | clase: rec.sport.hockey | términos: 82 -> 40 sin stop words
     doc  coseno               clase misma clase misma familia ya estaba antes                                términos que explican la similitud                                                                                                                                                   texto
0   5064   0.218    rec.sport.hockey          sí            sí              sí  games (33%), series (27%), watch (20%), playoff (15%), year (5%)  I only have one comment on this: You call this a *classic* playoff year and yet you don't include a Chicago-Detroit series. C'mon, I'm a Boston fan an
1   2645   0.168  rec.sport.baseball          no            sí              no                                           series (97%), time (3%)  These people were very silly. Any team that gets to the World Series can wi

#### Interpretación: efecto de eliminar stop words

| Doc | Clase | Misma clase (antes → después) | Misma familia (antes → después) | Vecinos que se mantienen |
|---|---|---|---|---|
| 10476 | rec.sport.hockey | 1 → 2 | 1 → 5 | 1 |
| 1824 | comp.sys.mac.hardware | 5 → 4 | 5 → 4 | 4 |
| 409 | comp.graphics | 5 → 4 | 5 → 5 | 4 |
| 4506 | rec.autos | 3 → 4 | 4 → 5 | 1 |
| 4012 | rec.sport.hockey | 3 → 5 | 4 → 5 | 3 |
| **Total** | | **17 → 19** | **20 → 24** | |

**Efecto sobre el vocabulario.** Solo se eliminan 309 de 101.631 términos, pero cada documento pierde cerca de la mitad de sus términos distintos (el 10476 pasa de 82 a 40). Las stop words son pocas como palabras distintas pero muy frecuentes dentro de cada texto. Además, 7 documentos más quedan vacíos (de 311 a 318), porque estaban compuestos únicamente por palabras funcionales.

**Contraste con las hipótesis.**

1. **El doc 10476 mejora, como se esperaba.** Los 5 vecinos pasan a ser de deportes, y aparece uno de hockey (10459) vinculado por *televised* y *coverage*, que es el tema real del documento: la cobertura televisiva de los playoffs. Los tres vecinos de béisbol comparten vocabulario genuino sobre ver partidos y series. En particular, *series* (97% de una similitud) designa tanto una serie de playoffs de hockey como la World Series de béisbol. Es un error razonable entre clases hermanas.
2. **Los docs 1824 y 409 se mantienen estables** (4 de 5 vecinos se conservan), como se esperaba, porque su similitud ya se apoyaba en términos distintivos. Sin embargo, ahora el coseno queda concentrado en una sola palabra (*pov* 100%, *tga* 100%, *processor* 99%).
3. **La predicción sobre la ambigüedad se cumplió solo en parte, y es el resultado más revelador.** Los vecinos erróneos por *peter* y *sound* desaparecen, aunque esas palabras no son stop words. Con el vectorizador anterior, esos vecinos también se sostenían en palabras funcionales (*are*, *those*, *who*; *to*, *the*, *it*), que aportaban cerca de un tercio del coseno. Sin ese apoyo, la palabra ambigua sola no alcanzó para mantenerlos entre los 5 primeros. Pero la ambigüedad no desapareció: surge un nuevo caso en el doc 1824, donde *processor* explica el 99% de la similitud con un aviso de venta de un **procesador de textos** (misc.forsale), en lugar de un procesador de computadora.

**Otras observaciones.** Los cosenos medios bajan (por ejemplo, de 0,217 a 0,165 en el doc 10476) mientras los vecinos mejoran, lo que confirma que el valor absoluto del coseno no mide la calidad de la similitud. El término *ve*, artefacto de tokenizar "I've", sigue aportando un 11% en el doc 4506, porque no está en la lista de stop words de sklearn: eliminarlas no corrige los problemas de tokenización.

**Conclusión del ejercicio.** Eliminar stop words mejora la similitud de forma clara (de 17 a 19 vecinos de la misma clase y de 20 a 24 de la misma familia), porque obliga al coseno a depender de términos con contenido. Corrige los errores que dependían del ruido gramatical, pero no las limitaciones de fondo de la bolsa de palabras. La polisemia persiste: *processor*, *series* o *peter* tienen más de un significado y el método no puede distinguirlos. Además, al concentrar la similitud en menos términos, el método queda más expuesto a que una sola palabra ambigua determine el resultado. Resolver ese problema requiere representaciones que tengan en cuenta el contexto en que aparece cada palabra.

### Ejercicio 2: Clasificador por prototipos

**Consigna:** Construir un modelo de clasificación por prototipos (tipo zero-shot). Clasificar los documentos del conjunto de test comparando cada uno con todos los de entrenamiento, y asignarle la etiqueta del documento de entrenamiento con mayor similaridad.

**Enfoque:**
- Cada documento de test se vectoriza con el TF-IDF ya ajustado en train (solo `transform`, para no filtrar información de test).
- Se calcula su similitud coseno contra los 11.314 documentos de train y se le asigna la clase del más similar. En términos técnicos, es un clasificador por vecino más cercano (k-NN con k = 1) con distancia coseno. No tiene etapa de entrenamiento: el "modelo" es el propio corpus de train. Lo llamamos *tipo* zero-shot porque no se ajusta ningún parámetro, aunque en sentido estricto usa ejemplos etiquetados.
- Probamos los dos vectorizadores del Ejercicio 1: el TF-IDF por defecto y el TF-IDF sin stop words.
- Para saber si el modelo realmente captura información, lo comparamos con dos líneas de base aleatorias: una que elige entre las 20 clases con igual probabilidad y otra que respeta la frecuencia de cada clase en train. También incluimos como referencia el Naïve Bayes de la plantilla.
- Métrica común: F1 macro sobre test, que da el mismo peso a las 20 clases.

In [30]:
# 2.1 – Clasificación y comparación con el azar
from sklearn.dummy import DummyClassifier

def clasificar_por_prototipo(X_te, X_tr, y_tr, bloque=1000):
    """
    Asigna a cada documento de test la clase del documento de train más similar
    (vecino más cercano por similitud coseno). Se calcula por bloques para no
    construir de una vez la matriz completa test x train.
    """
    pred = np.empty(X_te.shape[0], dtype=int)
    sim_max = np.empty(X_te.shape[0])
    for ini in range(0, X_te.shape[0], bloque):
        sims = cosine_similarity(X_te[ini:ini + bloque], X_tr)
        mejor = sims.argmax(axis=1)
        pred[ini:ini + bloque] = y_tr[mejor]
        sim_max[ini:ini + bloque] = sims.max(axis=1)
    return pred, sim_max

y_test = newsgroups_test.target

# Vectorizamos test con los vectorizadores ya ajustados en train (solo transform)
X_test = tfidfvect.transform(newsgroups_test.data)
X_test_sw = tfidf_sw.transform(newsgroups_test.data)

# Documentos de test vacíos: su vector es nulo y su coseno es 0 con todo train
vacios_test = X_test.getnnz(axis=1) == 0
print(f'Documentos vacíos en test: {vacios_test.sum()} de {X_test.shape[0]} ({vacios_test.mean():.1%})')

# Clasificador por prototipos con ambos vectorizadores
y_pred_proto, sim_proto = clasificar_por_prototipo(X_test, X_train, y_train)
y_pred_proto_sw, sim_proto_sw = clasificar_por_prototipo(X_test_sw, X_sw, y_train)

# Líneas de base aleatorias
azar_uniforme = DummyClassifier(strategy='uniform', random_state=42).fit(X_train, y_train)
azar_estratificado = DummyClassifier(strategy='stratified', random_state=42).fit(X_train, y_train)

resultados = pd.DataFrame([
    {'modelo': 'Azar uniforme (1 de 20 clases)',
     'F1 macro': f1_score(y_test, azar_uniforme.predict(X_test), average='macro')},
    {'modelo': 'Azar estratificado (según frecuencia de clases)',
     'F1 macro': f1_score(y_test, azar_estratificado.predict(X_test), average='macro')},
    {'modelo': 'Prototipos - TF-IDF por defecto',
     'F1 macro': f1_score(y_test, y_pred_proto, average='macro')},
    {'modelo': 'Prototipos - TF-IDF sin stop words',
     'F1 macro': f1_score(y_test, y_pred_proto_sw, average='macro')},
    {'modelo': 'Prototipos - TF-IDF por defecto (excluyendo test vacíos)',
     'F1 macro': f1_score(y_test[~vacios_test], y_pred_proto[~vacios_test], average='macro')},
    {'modelo': 'Naïve Bayes de la plantilla (referencia)',
     'F1 macro': 0.5854},
])
resultados['F1 macro'] = resultados['F1 macro'].round(4)
print(resultados.to_string(index=False))

# ¿Qué clase reciben los documentos vacíos?
print(f'\nClase asignada a los documentos vacíos: '
      f'{pd.Series(y_pred_proto[vacios_test]).map(lambda c: target_names[c]).value_counts().to_dict()}')

Documentos vacíos en test: 224 de 7532 (3.0%)
                                                  modelo  F1 macro
                          Azar uniforme (1 de 20 clases)    0.0499
         Azar estratificado (según frecuencia de clases)    0.0516
                         Prototipos - TF-IDF por defecto    0.5050
                      Prototipos - TF-IDF sin stop words    0.5327
Prototipos - TF-IDF por defecto (excluyendo test vacíos)    0.5171
                Naïve Bayes de la plantilla (referencia)    0.5854

Clase asignada a los documentos vacíos: {'rec.autos': 224}


#### Interpretación: desempeño del clasificador por prototipos

| Modelo | F1 macro |
|---|---|
| Azar uniforme | 0,050 |
| Azar estratificado | 0,052 |
| Prototipos – TF-IDF por defecto | 0,505 |
| Prototipos – TF-IDF por defecto (excluyendo test vacíos) | 0,517 |
| Prototipos – TF-IDF sin stop words | 0,533 |
| Naïve Bayes de la plantilla (referencia) | 0,585 |

**¿Supera al azar?** Claramente sí. Con 20 clases, una predicción aleatoria obtiene un F1 macro cercano a 0,05, tanto si elige las clases con igual probabilidad como si respeta su frecuencia en train. El clasificador por prototipos multiplica ese valor por 10, lo que muestra que la similitud coseno sobre TF-IDF captura información temática real: el documento más parecido de train suele pertenecer a la misma clase.

**Sesgo por documentos vacíos.** 224 documentos de test (3,0%) quedan vacíos al quitar encabezados, firmas y citas. Su similitud con todo train es 0, y `argmax` devuelve el primer índice, de modo que **los 224 reciben la clase rec.autos**, que es la del documento 0 de train. No es una decisión del modelo sino un efecto de la implementación, y perjudica en particular la precisión de rec.autos. Excluyendo esos documentos, el F1 macro sube de 0,505 a 0,517. En la práctica, para documentos sin contenido el clasificador no tiene información, y lo correcto sería abstenerse de predecir o tratarlos aparte.

**Efecto de las stop words.** Con el vectorizador sin stop words, el F1 macro sube de 0,505 a 0,533. Es coherente con lo observado en el Ejercicio 1: sin palabras funcionales, los vecinos más cercanos dependen de términos con contenido temático y pertenecen más seguido a la misma clase.

**Comparación con Naïve Bayes.** El clasificador por prototipos queda por debajo del Naïve Bayes de la plantilla (0,533 frente a 0,585), aunque este último no tiene ningún ajuste de parámetros. La diferencia se explica por cuántos documentos sostienen cada predicción. El prototipo decide con un solo vecino, así que cualquier coincidencia espuria como las del Ejercicio 1 (*peter*, *sound*, *processor*) se traduce directamente en un error de clasificación. Naïve Bayes, en cambio, estima la distribución de palabras de cada clase a partir de todos sus documentos, y el ruido de un documento individual se diluye. Además, el prototipo es costoso al predecir: compara cada documento nuevo contra los 11.314 de train, mientras que Naïve Bayes solo evalúa los parámetros de 20 clases.

In [31]:
# 2.2 – En qué falla y por qué?
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

pred = y_pred_proto_sw          # analizamos la mejor variante (sin stop words)
vacios_test_sw = X_test_sw.getnnz(axis=1) == 0

# 1) Desempeño por clase, ordenado de peor a mejor F1
prec, rec, f1, soporte = precision_recall_fscore_support(y_test, pred, zero_division=0)
por_clase = pd.DataFrame({
    'clase': target_names,
    'familia': [familia(c) for c in range(len(target_names))],
    'precision': prec.round(3),
    'recall': rec.round(3),
    'F1': f1.round(3),
    'docs test': soporte,
}).sort_values('F1')
print('DESEMPEÑO POR CLASE (de peor a mejor F1)')
print(por_clase.to_string(index=False))

# 2) Confusiones más frecuentes: ¿caen entre clases hermanas?
cm = confusion_matrix(y_test, pred)
np.fill_diagonal(cm, 0)
pares = [(cm[i, j], target_names[i], target_names[j], familia(i) == familia(j))
         for i in range(len(target_names)) for j in range(len(target_names)) if cm[i, j] > 0]
confusiones = pd.DataFrame(sorted(pares, reverse=True)[:12],
                           columns=['errores', 'clase real', 'clase predicha', 'misma familia'])
confusiones['misma familia'] = confusiones['misma familia'].map({True: 'sí', False: 'no'})
print('\nLAS 12 CONFUSIONES MÁS FRECUENTES')
print(confusiones.to_string(index=False))

errores = y_test != pred
misma_fam = np.array([familia(a) == familia(b) for a, b in zip(y_test, pred)])
print(f'\nTotal de errores: {errores.sum()} de {len(y_test)} ({errores.mean():.1%})')
print(f'Errores dentro de la misma familia: {(errores & misma_fam).sum()} ({(errores & misma_fam).sum() / errores.sum():.1%} de los errores)')

# 3) ¿La similitud con el vecino más cercano indica si la predicción es confiable?
tramos = pd.qcut(sim_proto_sw[~vacios_test_sw], q=5, duplicates='drop')
confianza = pd.DataFrame({
    'tramo de similitud máxima': tramos,
    'acierto': (~errores)[~vacios_test_sw],
}).groupby('tramo de similitud máxima', observed=True)['acierto'].agg(['count', 'mean'])
confianza.columns = ['docs', 'accuracy']
confianza['accuracy'] = confianza['accuracy'].round(3)
print('\nACCURACY SEGÚN LA SIMILITUD CON EL VECINO MÁS CERCANO (quintiles, sin documentos vacíos)')
print(confianza.to_string())

DESEMPEÑO POR CLASE (de peor a mejor F1)
                   clase     familia  precision  recall    F1  docs test
      talk.religion.misc    religión      0.340   0.331 0.335        251
      talk.politics.misc    política      0.429   0.358 0.390        310
         sci.electronics     ciencia      0.470   0.382 0.421        393
               rec.autos   vehículos      0.354   0.581 0.440        396
             alt.atheism    religión      0.461   0.442 0.451        319
      talk.politics.guns    política      0.506   0.484 0.494        364
           comp.graphics computación      0.500   0.494 0.497        389
   comp.sys.mac.hardware computación      0.482   0.532 0.506        385
 comp.os.ms-windows.misc computación      0.506   0.505 0.506        394
comp.sys.ibm.pc.hardware computación      0.488   0.541 0.513        392
            misc.forsale        misc      0.541   0.492 0.515        390
         rec.motorcycles   vehículos      0.548   0.555 0.552        398
          

In [32]:
print(pd.Series(y_test[vacios_test_sw]).map(lambda c: target_names[c]).value_counts().head(5))

rec.autos                  24
rec.sport.baseball         17
sci.crypt                  17
sci.space                  17
comp.os.ms-windows.misc    15
Name: count, dtype: int64


#### Interpretación: ¿en qué falla el clasificador por prototipos?

Analizamos la variante sin stop words (F1 macro 0,533), que se equivoca en el 46,2% de los documentos de test.

**Desempeño por clase.** Las clases con peor F1 son las residuales: **talk.religion.misc** (0,335) y **talk.politics.misc** (0,390). Por definición agrupan lo que no encaja en las otras clases de su tema, así que no tienen vocabulario propio y se superponen con sus clases hermanas. En el otro extremo están **rec.sport.hockey** (0,727) y **rec.sport.baseball** (0,648), cuyo vocabulario es muy distintivo: nombres de equipos, jugadores y términos del deporte. Es la misma conclusión del Ejercicio 1: el método funciona cuando una clase tiene términos que le son exclusivos.

**El caso de rec.autos.** Combina una precisión baja (0,354) con un recall relativamente alto (0,581), el patrón de una clase que recibe predicciones de más. Coincide con el sesgo identificado antes: todos los documentos de test vacíos se asignan a rec.autos. Lo verificamos contando la clase real de esos documentos, que se reparten entre todas las clases (por ejemplo, 17 de sci.space, 17 de sci.crypt y 17 de rec.sport.baseball). El sesgo distorsiona la clase en dos direcciones:
- **Precisión deprimida:** los documentos vacíos de otras clases son falsos positivos de rec.autos. Por ejemplo, 17 de los 38 errores sci.space → rec.autos (45%) son documentos vacíos, no confusiones temáticas.
- **Recall inflado:** 24 documentos vacíos son efectivamente de rec.autos y se cuentan como aciertos, aunque el modelo no tuvo ninguna información para clasificarlos.

Las métricas de rec.autos reflejan en parte un artefacto de implementación y no el comportamiento real del clasificador.

**Confusiones entre clases hermanas.** Las confusiones más frecuentes ocurren dentro de una misma familia temática: 9 de las 12 principales. Las mayores son talk.politics.misc → talk.politics.guns (58), rec.motorcycles → rec.autos (52), comp.windows.x → comp.graphics (50), comp.sys.mac.hardware → comp.sys.ibm.pc.hardware (45) y alt.atheism → soc.religion.christian (42). Son clases que comparten vocabulario, y el vecino más cercano puede pertenecer indistintamente a una u otra. Sin embargo, en el total solo el 31,2% de los errores cae dentro de la misma familia. Esto tiene dos explicaciones:
- La agrupación en familias es aproximada. Confusiones como sci.electronics → comp.sys.ibm.pc.hardware (41) o misc.forsale → comp.sys.ibm.pc.hardware (37) cruzan familias, pero comparten vocabulario de hardware, ya que muchos avisos de venta son de computadoras.
- Existe una cola larga de errores dispersos entre clases sin relación. Al depender de un único vecino, cualquier coincidencia puntual de palabras, como las observadas en el Ejercicio 1 con *peter*, *sound* o *processor*, puede asignar el documento a una clase arbitraria.

**La similitud como medida de confianza.**

| Quintil de similitud máxima | Accuracy |
|---|---|
| 0,07 – 0,18 | 0,386 |
| 0,18 – 0,23 | 0,453 |
| 0,23 – 0,27 | 0,549 |
| 0,27 – 0,35 | 0,619 |
| 0,35 – 1,00 | 0,755 |

La accuracy crece de forma monótona con la similitud del vecino más cercano: casi se duplica entre el quintil más bajo y el más alto. Esto matiza la observación del Ejercicio 1. Para un documento individual el valor del coseno no garantiza que el vecino sea correcto, pero en promedio sí es una señal de confianza. Cuanto más parecido es el vecino, más probable es que comparta la clase. Esto tiene un uso práctico: se podría fijar un umbral de similitud por debajo del cual el modelo no predice y deriva el documento a revisión manual. Aun en el quintil más bajo, la accuracy (0,386) supera ampliamente al azar.

**Conclusión del ejercicio.** Un clasificador sin entrenamiento, basado solo en el documento más similar, alcanza un F1 macro de 0,533, diez veces mejor que el azar. Esto confirma que TF-IDF captura información temática. Sus limitaciones son tres. Depende de un único vecino, por lo que traslada a la clasificación todas las coincidencias espurias de la bolsa de palabras. No separa bien las clases que comparten vocabulario, en especial las residuales. Y no puede tratar documentos sin contenido, que se asignan de forma arbitraria. Un modelo que agregue la información de todos los documentos de cada clase, como Naïve Bayes, debería ser más robusto frente a ese ruido. Lo evaluamos en el Ejercicio 3.

### Ejercicio 3: Optimización de clasificadores Naïve Bayes

**Consigna:** Entrenar modelos de clasificación Naïve Bayes para maximizar el F1 macro en el conjunto de test. Considerar cambiar parámetros de instanciación del vectorizador y de los modelos, y probar los modelos Multinomial y ComplementNB. **No modificar el hiperparámetro `ngram_range`.**

**Punto de partida.** El modelo de la plantilla (`TfidfVectorizer()` + `MultinomialNB()`, todo por defecto) obtiene un F1 macro de **0,585** en test. Es nuestra línea base.

**Metodología.**
- Armamos un `Pipeline` con vectorizador y clasificador, y buscamos la mejor combinación de hiperparámetros con `GridSearchCV`, usando validación cruzada estratificada de 5 folds **solo sobre train**. El conjunto de test se usa una única vez, para medir los modelos elegidos. Si eligiéramos los parámetros mirando el F1 de test, el resultado final sería optimista, porque estaríamos ajustando el modelo al conjunto de evaluación.
- `ngram_range` queda en su valor por defecto, `(1, 1)`, es decir, unigramas, como exige la consigna.

**Hiperparámetros explorados y su motivación:**

| Hiperparámetro | Valores | Qué controla |
|---|---|---|
| `stop_words` | `None`, `'english'` | Elimina palabras funcionales. En los Ejercicios 1 y 2 vimos que introducen ruido en la similitud. |
| `min_df` | 1, 2, 5 | Descarta términos que aparecen en menos de *n* documentos: errores de tipeo, nombres raros, artefactos. Reduce el vocabulario y el sobreajuste a términos únicos. |
| `max_df` | 0,5 y 1,0 | Descarta términos que aparecen en más de la mitad de los documentos, que discriminan poco entre clases. |
| `sublinear_tf` | `False`, `True` | Reemplaza la frecuencia del término por 1 + log(tf). Evita que una palabra repetida muchas veces en un documento domine su vector. |
| Modelo | `MultinomialNB`, `ComplementNB` | ComplementNB estima los pesos de cada clase a partir de los documentos de **todas las demás clases** (su "complemento"). Así obtiene estimaciones más estables y menos sesgadas hacia las clases con más datos. |
| `alpha` | 0,01 a 1,0 | Suavizado de Laplace: cuánta probabilidad se asigna a palabras no vistas en una clase. Valores bajos hacen que el modelo confíe más en las frecuencias observadas. |

In [33]:
# 3.1 – Búsqueda de hiperparámetros con validación cruzada
import time
from tempfile import mkdtemp
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Pipeline vectorizador + clasificador. 'memory' guarda en caché el vectorizador ajustado,
# así no se reajusta para cada combinación de modelo y alpha (acelera mucho la búsqueda).
# ngram_range NO se incluye en la grilla: queda en su valor por defecto (1, 1), como pide la consigna.
pipe = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', MultinomialNB()),
], memory=mkdtemp())

grilla_vectorizador = {
    'tfidf__stop_words': [None, 'english'],
    'tfidf__min_df': [1, 2, 5],
    'tfidf__max_df': [0.5, 1.0],
    'tfidf__sublinear_tf': [False, True],
}
grilla = [
    {**grilla_vectorizador, 'clf': [MultinomialNB()], 'clf__alpha': [0.01, 0.03, 0.1, 0.3, 1.0]},
    {**grilla_vectorizador, 'clf': [ComplementNB()], 'clf__alpha': [0.01, 0.03, 0.1, 0.3, 1.0]},
]

# Selección de hiperparámetros con validación cruzada SOLO sobre train.
# El conjunto de test se usa una única vez, para medir el modelo elegido.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
busqueda = GridSearchCV(pipe, grilla, scoring='f1_macro', cv=cv, n_jobs=-1)

inicio = time.time()
busqueda.fit(newsgroups_train.data, y_train)
print(f'Búsqueda completada en {(time.time() - inicio) / 60:.1f} minutos '
      f'({len(busqueda.cv_results_["params"])} combinaciones x 5 folds)')

# Resultados de la validación cruzada, ordenados
cv_res = pd.DataFrame(busqueda.cv_results_)
cv_res['modelo'] = cv_res['param_clf'].map(lambda m: type(m).__name__)
cols = ['modelo', 'param_clf__alpha', 'param_tfidf__stop_words', 'param_tfidf__min_df',
        'param_tfidf__max_df', 'param_tfidf__sublinear_tf', 'mean_test_score', 'std_test_score']
tabla_cv = cv_res[cols].sort_values('mean_test_score', ascending=False)
tabla_cv.columns = ['modelo', 'alpha', 'stop_words', 'min_df', 'max_df', 'sublinear_tf', 'F1 CV', 'desvío CV']
tabla_cv[['F1 CV', 'desvío CV']] = tabla_cv[['F1 CV', 'desvío CV']].round(4)
tabla_cv['stop_words'] = tabla_cv['stop_words'].map(lambda v: 'english' if v == 'english' else 'ninguna')

print('\nTOP 10 CONFIGURACIONES (F1 macro en validación cruzada)')
print(tabla_cv.head(10).to_string(index=False))

# Mejor configuración de cada modelo, evaluada en test
finales = []
for nombre in ['MultinomialNB', 'ComplementNB']:
    mejor_fila = cv_res[cv_res['modelo'] == nombre].sort_values('mean_test_score', ascending=False).iloc[0]
    modelo = pipe.set_params(**mejor_fila['params']).fit(newsgroups_train.data, y_train)
    y_pred_nb = modelo.predict(newsgroups_test.data)
    finales.append({
        'modelo': nombre,
        'parámetros': {k.split('__')[-1]: v for k, v in mejor_fila['params'].items() if k != 'clf'},
        'F1 CV': round(mejor_fila['mean_test_score'], 4),
        'F1 test': round(f1_score(y_test, y_pred_nb, average='macro'), 4),
    })

print('\nMEJOR CONFIGURACIÓN DE CADA MODELO')
print(pd.DataFrame(finales).to_string(index=False))

Búsqueda completada en 16.5 minutos (240 combinaciones x 5 folds)

TOP 10 CONFIGURACIONES (F1 macro en validación cruzada)
      modelo  alpha stop_words  min_df  max_df  sublinear_tf  F1 CV  desvío CV
ComplementNB    0.3    ninguna       1     1.0          True 0.7634     0.0037
ComplementNB    0.3    ninguna       1     0.5          True 0.7631     0.0040
ComplementNB    0.1    ninguna       1     1.0          True 0.7626     0.0053
ComplementNB    0.3    english       1     1.0          True 0.7624     0.0038
ComplementNB    0.3    english       1     0.5          True 0.7624     0.0038
ComplementNB    0.1    ninguna       1     0.5          True 0.7622     0.0054
ComplementNB    0.1    ninguna       1     1.0         False 0.7619     0.0039
ComplementNB    0.3    ninguna       1     0.5         False 0.7616     0.0052
ComplementNB    0.3    english       1     0.5         False 0.7613     0.0042
ComplementNB    0.3    english       1     1.0         False 0.7613     0.0042

MEJOR C

#### Interpretación: búsqueda de hiperparámetros

| Modelo | Mejores parámetros | F1 CV | F1 test |
|---|---|---|---|
| Plantilla (referencia) | todo por defecto, `alpha=1` | — | 0,585 |
| MultinomialNB | `alpha=0.01`, `stop_words='english'`, `max_df=0.5`, `min_df=1` | 0,757 | 0,684 |
| **ComplementNB** | `alpha=0.3`, `sublinear_tf=True`, `min_df=1`, sin stop words | **0,763** | **0,699** |

**Mejora obtenida.** El mejor modelo, ComplementNB, alcanza un F1 macro de 0,699 en test: 11,4 puntos por encima de la plantilla y 16,6 por encima del clasificador por prototipos. Se obtuvo con unigramas, respetando la restricción sobre `ngram_range`.

**Las diferencias entre las mejores configuraciones no son significativas.** Las 10 mejores tienen un F1 de validación cruzada entre 0,7613 y 0,7634. Esa diferencia (0,002) es menor que el desvío estándar entre folds (0,004 a 0,005), así que se trata de una meseta de configuraciones equivalentes y no de un óptimo claro.

**Observaciones sobre los hiperparámetros.**
- **Modelo.** Las 10 mejores configuraciones son de ComplementNB, aunque su ventaja sobre MultinomialNB es moderada (0,763 frente a 0,757 en CV y 0,699 frente a 0,684 en test). Como las clases de 20 Newsgroups están bastante balanceadas, la ventaja no se explica por el desbalance sino por la forma de estimar los pesos: ComplementNB usa los documentos de todas las demás clases, lo que da estimaciones más estables.
- **Suavizado (`alpha`).** El valor óptimo es bajo en ambos modelos (0,01 y 0,3), muy por debajo del valor por defecto (1,0). Con TF-IDF, los pesos de cada término son valores fraccionarios pequeños. Un suavizado de 1 sobre un vocabulario de más de 100.000 términos domina esas frecuencias observadas y acerca las distribuciones de todas las clases a la uniforme.
- **Stop words.** A diferencia de los Ejercicios 1 y 2, eliminarlas no mejora el resultado: el mejor ComplementNB las conserva. Nuestra hipótesis es que la diferencia está en cómo cada método usa las palabras. En la similitud coseno, una stop word compartida suma directamente al puntaje entre dos documentos. En Naïve Bayes, una palabra con frecuencia similar en todas las clases suma aproximadamente lo mismo al puntaje de cada clase y no cambia cuál resulta elegida.
- **Términos raros (`min_df`).** Las mejores configuraciones conservan todos los términos (`min_df=1`). Términos poco frecuentes como nombres de jugadores, de programas o de modelos de procesador son muy indicativos de una clase, y descartarlos pierde información.
- **`max_df` y `sublinear_tf`** tienen efectos menores, dentro del margen de variación entre folds.

**Brecha entre validación cruzada y test.** El F1 cae de 0,763 en CV a 0,699 en test, y la caída es similar en ambos modelos, por lo que no se debe a la selección de hiperparámetros. La explicación más probable es que la división de 20 Newsgroups es temporal: test contiene mensajes posteriores a los de train. En la validación cruzada, los folds mezclan mensajes de los mismos hilos de discusión, que comparten temas y nombres propios, mientras que test incluye temas y conversaciones nuevas. La estimación por CV es, por lo tanto, algo optimista respecto del desempeño sobre datos futuros.

En la siguiente sección cuantificamos cuánto aporta cada decisión mediante un análisis de ablación.

#### Análisis de ablación: ¿cuánto aporta cada decisión?

Partimos de la plantilla y avanzamos hasta el mejor modelo (configuraciones 1 a 5). Después, sobre el mejor ComplementNB, cambiamos **un solo parámetro por vez** (5a a 5d) para aislar su efecto. Reportamos el F1 en validación cruzada y en test.

In [34]:
# 3.2 – Ablación
from sklearn.model_selection import cross_val_score

def evaluar(nombre, vect_params, modelo):
    """F1 macro en validación cruzada (train) y en test para una configuración."""
    p = Pipeline([('tfidf', TfidfVectorizer(**vect_params)), ('clf', modelo)])
    f1_cv = cross_val_score(p, newsgroups_train.data, y_train, cv=cv, scoring='f1_macro', n_jobs=-1).mean()
    p.fit(newsgroups_train.data, y_train)
    f1_te = f1_score(y_test, p.predict(newsgroups_test.data), average='macro')
    return {'configuración': nombre, 'F1 CV': round(f1_cv, 4), 'F1 test': round(f1_te, 4)}

mejor_cnb = dict(stop_words=None, min_df=1, max_df=1.0, sublinear_tf=True)

experimentos = [
    # Camino desde la plantilla
    ('1. Plantilla: TF-IDF por defecto + MultinomialNB(alpha=1)', {}, MultinomialNB()),
    ('2. Plantilla con alpha=0.01', {}, MultinomialNB(alpha=0.01)),
    ('3. Mejor MultinomialNB (alpha=0.01, stop words, max_df=0.5)',
     dict(stop_words='english', max_df=0.5), MultinomialNB(alpha=0.01)),
    ('4. ComplementNB con todo por defecto (alpha=1)', {}, ComplementNB()),
    ('5. Mejor ComplementNB (alpha=0.3, sublinear_tf)', mejor_cnb, ComplementNB(alpha=0.3)),
    # Ablación: al mejor ComplementNB le cambiamos un parámetro por vez
    ('5a. Mejor CNB sin sublinear_tf', {**mejor_cnb, 'sublinear_tf': False}, ComplementNB(alpha=0.3)),
    ('5b. Mejor CNB con alpha=1', mejor_cnb, ComplementNB(alpha=1.0)),
    ('5c. Mejor CNB con stop words', {**mejor_cnb, 'stop_words': 'english'}, ComplementNB(alpha=0.3)),
    ('5d. Mejor CNB con min_df=5', {**mejor_cnb, 'min_df': 5}, ComplementNB(alpha=0.3)),
]

inicio = time.time()
ablacion = pd.DataFrame([evaluar(*e) for e in experimentos])
print(f'Completado en {(time.time() - inicio) / 60:.1f} minutos\n')
ablacion['dif. test vs plantilla'] = (ablacion['F1 test'] - ablacion.loc[0, 'F1 test']).round(4)
print(ablacion.to_string(index=False))

Completado en 2.2 minutos

                                              configuración  F1 CV  F1 test  dif. test vs plantilla
  1. Plantilla: TF-IDF por defecto + MultinomialNB(alpha=1) 0.6234   0.5854                  0.0000
                                2. Plantilla con alpha=0.01 0.7549   0.6829                  0.0975
3. Mejor MultinomialNB (alpha=0.01, stop words, max_df=0.5) 0.7574   0.6844                  0.0990
             4. ComplementNB con todo por defecto (alpha=1) 0.7489   0.6930                  0.1076
            5. Mejor ComplementNB (alpha=0.3, sublinear_tf) 0.7634   0.6990                  0.1136
                             5a. Mejor CNB sin sublinear_tf 0.7611   0.6999                  0.1145
                                  5b. Mejor CNB con alpha=1 0.7502   0.6920                  0.1066
                               5c. Mejor CNB con stop words 0.7624   0.6979                  0.1125
                                 5d. Mejor CNB con min_df=5 0.7274   0.68

#### Interpretación: análisis de ablación

| Configuración | F1 CV | F1 test | Diferencia en test vs. plantilla |
|---|---|---|---|
| 1. Plantilla: MultinomialNB, `alpha=1` | 0,623 | 0,585 | — |
| 2. Plantilla con `alpha=0.01` | 0,755 | 0,683 | +9,8 |
| 3. Mejor MultinomialNB | 0,757 | 0,684 | +9,9 |
| 4. ComplementNB por defecto (`alpha=1`) | 0,749 | 0,693 | +10,8 |
| **5. Mejor ComplementNB** | **0,763** | **0,699** | **+11,4** |
| 5a. Mejor CNB sin `sublinear_tf` | 0,761 | 0,700 | +11,5 |
| 5b. Mejor CNB con `alpha=1` | 0,750 | 0,692 | +10,7 |
| 5c. Mejor CNB con stop words | 0,762 | 0,698 | +11,3 |
| 5d. Mejor CNB con `min_df=5` | 0,727 | 0,681 | +9,6 |

**El suavizado explica casi toda la mejora en MultinomialNB.** Bajar `alpha` de 1 a 0,01, sin modificar el vectorizador, sube el F1 de test de 0,585 a 0,683: el 86% de toda la mejora obtenida. Los demás ajustes de MultinomialNB (stop words y `max_df`) aportan apenas 0,15 puntos adicionales. El problema del modelo de la plantilla no era la representación del texto sino un suavizado excesivo, que acercaba las distribuciones de palabras de todas las clases a la uniforme.

**ComplementNB es robusto al suavizado.** Con todos sus parámetros por defecto, incluido `alpha=1`, ComplementNB ya alcanza 0,693 en test, y en el mejor modelo volver a `alpha=1` cuesta solo 0,7 puntos. La explicación está en cómo estima sus pesos. MultinomialNB estima la distribución de cada clase solo con sus propios documentos (unos 600), y el suavizado pesa mucho frente a esa cantidad de datos. ComplementNB estima cada clase a partir de los documentos de las otras 19 (unos 10.700), y frente a esa masa de datos el mismo suavizado tiene un efecto mucho menor. Esto explica por qué supera a MultinomialNB sin ajuste y por qué su ventaja se reduce cuando MultinomialNB tiene el `alpha` adecuado.

**Las stop words no influyen en Naïve Bayes.** Agregarlas al mejor modelo cambia el F1 en apenas −0,001, tanto en validación cruzada como en test. Se confirma la hipótesis planteada: una palabra funcional aparece con frecuencia similar en todas las clases, suma aproximadamente lo mismo al puntaje de cada una y no altera cuál resulta elegida. Esto contrasta con los Ejercicios 1 y 2, donde las stop words sí distorsionaban la similitud entre documentos, porque allí sumaban directamente al puntaje.

**Los términos poco frecuentes son valiosos.** Exigir que cada término aparezca en al menos 5 documentos (`min_df=5`) es la modificación que más perjudica al mejor modelo: −3,6 puntos en validación cruzada y −1,8 en test. Los términos raros incluyen nombres de jugadores, programas, modelos de hardware y términos técnicos muy específicos de una clase. Descartarlos elimina parte de la información más discriminativa.

**El efecto de `sublinear_tf` no es significativo.** Quitarlo baja el F1 de validación cruzada en 0,2 puntos y lo sube en test en 0,1. Ambas diferencias están dentro de la variación entre folds. Aunque la configuración 5a obtiene un F1 de test ligeramente mayor, mantenemos el modelo elegido por validación cruzada, ya que elegir mirando el resultado de test haría que esa métrica dejara de ser una estimación honesta del desempeño.

#### Conclusión del Ejercicio 3

| Modelo | F1 macro (test) |
|---|---|
| Azar uniforme | 0,050 |
| Clasificador por prototipos (sin stop words) | 0,533 |
| Naïve Bayes de la plantilla | 0,585 |
| **ComplementNB optimizado** | **0,699** |

El mejor modelo, ComplementNB con `alpha=0.3`, alcanza un F1 macro de 0,699 en test. Supera en 11,4 puntos a la plantilla y en 16,6 al clasificador por prototipos, sin modificar `ngram_range`. El análisis muestra que la mejora se concentra en dos decisiones: **el suavizado** (bajar `alpha`, o usar ComplementNB, que es poco sensible a él) y **conservar los términos poco frecuentes**. El resto de los ajustes, incluida la eliminación de stop words, tiene efectos dentro del margen de variación.

Naïve Bayes supera al clasificador por prototipos porque cada predicción se apoya en la distribución de palabras de toda la clase, estimada con cientos de documentos, y no en un único vecino. Así, las coincidencias espurias de palabras que observamos en los Ejercicios 1 y 2 se diluyen en lugar de decidir el resultado.

### Ejercicio 4: Similaridad entre palabras (matriz término-documento)

**Consigna:** Transponer la matriz documento-término para obtener una matriz término-documento, que puede interpretarse como una colección de vectores de palabras. Estudiar la similaridad entre palabras tomando 5 palabras y sus 5 más similares. Las palabras se eligen manualmente para evitar términos poco interpretables.

**Idea.** En la matriz documento-término, cada fila es un documento descrito por las palabras que contiene. Al transponerla, cada fila pasa a ser una **palabra descrita por los documentos en que aparece**: un vector de 11.314 dimensiones con su peso TF-IDF en cada documento. Dos palabras resultan similares si aparecen en los mismos documentos con pesos parecidos, es decir, por **co-ocurrencia**. Es la hipótesis distribucional, según la cual el significado de una palabra se refleja en los contextos en que aparece. Es la misma idea que después desarrollan los word embeddings, aunque aquí el contexto es el documento completo.

**Palabras elegidas.** Combino palabras de tema claro con palabras ambiguas, para observar dónde la co-ocurrencia captura relaciones de significado y dónde se confunde:

| Palabra | Tipo | Motivo de la elección |
|---|---|---|
| *car* | tema claro | vocabulario automotor, concentrado en rec.autos |
| *hockey* | tema claro | vocabulario deportivo muy distintivo |
| *god* | tema claro | religión, repartida entre tres clases hermanas |
| *space* | ambigua | espacio exterior (sci.space) o espacio en disco (computación) |
| *drive* | ambigua | conducir un vehículo o unidad de disco |

**Diseño.** Calculamos las similitudes con dos matrices:
1. la del **TF-IDF por defecto** (la del Ejercicio 1);
2. una **filtrada**: sin stop words y descartando términos que aparecen en menos de 5 documentos (`min_df=5`).

Para cada palabra similar informo en cuántos documentos aparece y en qué clase concentra su peso. Esto permite identificar coincidencias espurias y ver hacia qué significado se inclina cada palabra ambigua.

In [35]:
# 4.1 – Construcción de las matrices término-documento
def preparar_terminos(X, vectorizador):
    """
    A partir de una matriz documento-término devuelve:
    - XT: matriz término-documento (cada fila es el vector de una palabra)
    - vocab / idx2w: palabra -> índice e índice -> palabra
    - df: en cuántos documentos aparece cada palabra
    - reparto: qué proporción del peso TF-IDF de cada palabra cae en cada clase
    """
    XT = X.T.tocsr()
    vocab = vectorizador.vocabulary_
    idx2w = {v: k for k, v in vocab.items()}
    df = X.getnnz(axis=0)
    Y = np.eye(len(target_names))[y_train]          # one-hot documentos x clases
    masa = np.asarray(XT @ Y)                         # palabras x clases
    reparto = masa / np.maximum(masa.sum(axis=1, keepdims=True), 1e-12)
    return XT, vocab, idx2w, df, reparto

def clase_principal(reparto, i):
    c = reparto[i].argmax()
    return f'{target_names[c]} ({reparto[i, c]:.0%})'

def palabras_similares(palabra, XT, vocab, idx2w, df, reparto, k=5):
    if palabra not in vocab:
        print(f'"{palabra}" no está en el vocabulario de este vectorizador')
        return None
    i = vocab[palabra]
    cos = cosine_similarity(XT[i], XT)[0]
    top = [j for j in np.argsort(cos)[::-1] if j != i][:k]
    return pd.DataFrame({
        'palabra similar': [idx2w[j] for j in top],
        'coseno': cos[top].round(3),
        'docs en que aparece': df[top],
        'clase principal': [clase_principal(reparto, j) for j in top],
    })

# Matriz término-documento con el vectorizador por defecto (el del Ejercicio 1)
terminos_base = preparar_terminos(X_train, tfidfvect)

# Matriz término-documento con stop words eliminadas y sin términos muy raros
tfidf_filt = TfidfVectorizer(stop_words='english', min_df=5)
X_filt = tfidf_filt.fit_transform(newsgroups_train.data)
terminos_filt = preparar_terminos(X_filt, tfidf_filt)

print(f'Matriz término-documento por defecto: {terminos_base[0].shape} (palabras x documentos)')
print(f'Matriz término-documento filtrada:    {terminos_filt[0].shape}')

Matriz término-documento por defecto: (101631, 11314) (palabras x documentos)
Matriz término-documento filtrada:    (17797, 11314)


In [36]:
# 4.2 – Las 5 palabras más similares
palabras_elegidas = ['car', 'hockey', 'god', 'space', 'drive']

for palabra in palabras_elegidas:
    print('=' * 100)
    XT, vocab, idx2w, df, reparto = terminos_base
    if palabra in vocab:
        i = vocab[palabra]
        top3 = reparto[i].argsort()[::-1][:3]
        print(f'PALABRA: {palabra.upper()} | aparece en {df[i]} documentos | '
              f'clases donde más pesa: ' + ', '.join(f'{target_names[c]} ({reparto[i, c]:.0%})' for c in top3))
    for nombre, terminos in [('TF-IDF por defecto', terminos_base),
                             ('TF-IDF sin stop words y min_df=5', terminos_filt)]:
        tabla = palabras_similares(palabra, *terminos)
        if tabla is not None:
            print(f'\n  {nombre}')
            print(tabla.to_string(index=False))

PALABRA: CAR | aparece en 396 documentos | clases donde más pesa: rec.autos (68%), rec.motorcycles (9%), misc.forsale (7%)

  TF-IDF por defecto
palabra similar  coseno  docs en que aparece  clase principal
           cars   0.180                  177  rec.autos (72%)
      criterium   0.177                    1 rec.autos (100%)
          civic   0.175                   22  rec.autos (79%)
          owner   0.169                   99  rec.autos (25%)
         dealer   0.168                  107  rec.autos (57%)

  TF-IDF sin stop words y min_df=5
palabra similar  coseno  docs en que aparece clase principal
           cars   0.192                  177 rec.autos (72%)
         dealer   0.177                  107 rec.autos (57%)
          civic   0.163                   22 rec.autos (79%)
           loan   0.156                   14 rec.autos (39%)
          owner   0.148                   99 rec.autos (26%)
PALABRA: HOCKEY | aparece en 150 documentos | clases donde más pesa: rec.sport.ho

#### Interpretación: similaridad entre palabras

**Qué relaciones captura.** La similitud por co-ocurrencia en documentos recupera relaciones con sentido para las palabras de tema claro:
- **Variantes de una misma palabra:** *car* → *cars*, *drive* → *drives*. Las agrupa sin ningún procesamiento morfológico, solo porque aparecen en los mismos documentos.
- **Expresiones compuestas:** *hard* es una palabra genérica (su clase principal concentra apenas el 17% de su peso), pero es la segunda más similar a *drive* porque ambas forman "hard drive". Aunque trabajamos con unigramas, la co-ocurrencia recupera parte de la información de las expresiones de dos palabras.
- **Asociación temática:** *car* → *dealer*, *owner*, *loan*; *god* → *jesus*, *bible*, *christ*, *faith*; *hockey* → *nhl*, *ncaa*, *league*, *players*; *space* → *nasa*, *shuttle*, *exploration*. *existence* aparece entre las similares a *god* por los debates sobre la existencia de Dios en alt.atheism.

Las palabras similares no son sinónimos sino términos del mismo tema. Esto es consecuencia de usar el documento completo como contexto: dos palabras son similares si se usan para hablar de lo mismo, aunque no signifiquen lo mismo.

**Efecto de los términos raros.** Con el vectorizador por defecto aparecen términos que figuran en uno a cuatro documentos: *criterium* (1), *xenophobes* (1), *affiliates* (2), *seds* (3), *enfant* (4), *seti* (4). El motivo es estructural. El vector de una palabra que aparece en un solo documento tiene un único valor distinto de cero, y su coseno con otra palabra depende solo del peso de esa otra palabra en ese documento. Cualquier término único de un documento que habla mucho de hockey resulta muy similar a *hockey*, aunque no tenga relación con el tema (*xenophobes*). Al exigir que cada término aparezca en al menos 5 documentos (`min_df=5`), estos casos desaparecen y los reemplazan palabras con una relación genuina (*players*, *league*, *exploration*, *aeronautics*, *loan*). El umbral es arbitrario: *cfa*, que aparece en exactamente 5 documentos, sobrevive al filtro. El filtrado reduce el problema pero no lo elimina.

Este resultado contrasta con el Ejercicio 3, donde los términos raros resultaron valiosos para clasificar. Para un clasificador, una palabra que aparece en pocos documentos de una misma clase es una señal fuerte. Para comparar palabras entre sí, en cambio, su vector se apoya en tan pocos documentos que la similitud queda determinada por coincidencias.

**Efecto de las stop words.** Con el vectorizador por defecto, *that* (presente en 6.580 documentos) aparece entre las más similares a *god*, probablemente porque los textos religiosos son debates extensos y argumentativos. Al eliminar las stop words la reemplaza *faith*.

**Palabras ambiguas: domina el significado mayoritario.**
- *space* concentra el 72% de su peso en sci.space, y todas sus similares pertenecen al espacio exterior. El sentido de "espacio en disco", presente en las clases de computación (5% en comp.os.ms-windows.misc y 4% en comp.graphics), no se refleja en ninguna.
- *drive* concentra su peso en las clases de hardware, y todas sus similares (*drives*, *floppy*, *hard*, *disk*, *ide*) corresponden a "unidad de disco". El sentido de "conducir" desaparece: rec.autos ni siquiera figura entre sus tres clases principales.

La causa es que esta representación asigna **un único vector a cada palabra**, que mezcla todos sus usos en el corpus. El significado más frecuente se impone y los minoritarios quedan ocultos. Es la misma limitación observada en el Ejercicio 1 con *peter*, *sound* o *processor*, ahora vista desde las palabras en lugar de los documentos.

**Conclusión del ejercicio.** Transponer la matriz documento-término produce vectores de palabras que capturan asociaciones temáticas razonables, siempre que se filtren los términos raros y las palabras funcionales. Tiene tres limitaciones. La primera es la dimensionalidad: cada palabra es un vector de 11.314 dimensiones, casi todas nulas, que crece con el corpus. La segunda es que, al usar el documento completo como contexto, captura temas y no significados. La tercera es que asigna un solo vector a cada palabra, sin distinguir sus sentidos. Los word embeddings de la clase 2 (como Word2Vec, GloVe o FastText) atacan las dos primeras: representan cada palabra con un vector denso de pocas dimensiones, aprendido a partir de ventanas de pocas palabras. La tercera requiere modelos contextuales, que asignan a cada aparición de una palabra un vector distinto según la oración en que aparece.

#### Bonus: ¿el sentido minoritario existe o desapareció?

En el corpus completo, *drive* solo mostró su sentido de "unidad de disco" y *space* solo el de "espacio exterior". Queremos saber si los otros sentidos ("conducir" y "espacio en disco") no aparecen en los datos o si existen pero quedan ocultos detrás del significado mayoritario.

Para comprobarlo recalculamos la matriz término-documento usando **solo los documentos de las clases donde esperamos el sentido minoritario**:
- *drive* en rec.autos y rec.motorcycles;
- *space* en las cinco clases de computación.

Si las palabras similares cambian hacia el otro sentido, queda demostrado que el significado depende del contexto en que se usa la palabra, y que el vector único del corpus completo lo ocultaba. Usamos la matriz filtrada (sin stop words, `min_df=5`), y dentro de cada subcorpus exigimos que las palabras candidatas aparezcan en al menos 3 documentos, para no reintroducir el problema de los términos raros.

In [38]:
# 4.3 – Similares dentro de un subcorpus
def similares_en_subcorpus(palabra, clases, X=X_filt, vectorizador=tfidf_filt, k=5, min_docs=3):
    """
    Palabras más similares a 'palabra' usando solo los documentos de las clases indicadas.
    Se exige que cada palabra candidata aparezca en al menos 'min_docs' documentos del
    subcorpus, para evitar el problema de los términos raros visto antes.
    """
    ids = [target_names.index(c) for c in clases]
    docs = np.isin(y_train, ids)
    XT_sub = X[docs].T.tocsr()                       # palabras x documentos del subcorpus
    df_sub = X[docs].getnnz(axis=0)
    idx2w = {v: w for w, v in vectorizador.vocabulary_.items()}
    i = vectorizador.vocabulary_[palabra]

    cos = cosine_similarity(XT_sub[i], XT_sub)[0]
    cos[df_sub < min_docs] = -1                      # descartamos términos raros del subcorpus
    cos[i] = -1                                      # y la propia palabra
    top = np.argsort(cos)[::-1][:k]

    print(f'"{palabra}" en {", ".join(clases)}: {docs.sum()} documentos, '
          f'la palabra aparece en {df_sub[i]}')
    return pd.DataFrame({
        'palabra similar': [idx2w[j] for j in top],
        'coseno': cos[top].round(3),
        'docs del subcorpus': df_sub[top],
    })

casos = [
    ('drive', ['rec.autos', 'rec.motorcycles']),
    ('space', ['comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware',
               'comp.sys.mac.hardware', 'comp.windows.x']),
]

for palabra, clases in casos:
    print('=' * 100)
    print(similares_en_subcorpus(palabra, clases).to_string(index=False))
    print(f'\n  Comparación: en el corpus completo -> '
          f'{", ".join(palabras_similares(palabra, *terminos_filt)["palabra similar"])}')

"drive" en rec.autos, rec.motorcycles: 1192 documentos, la palabra aparece en 81
palabra similar  coseno  docs del subcorpus
        talking   0.391                  21
        wheelie   0.342                   6
          shaft   0.330                  15
        address   0.208                  17
           drag   0.207                   5

  Comparación: en el corpus completo -> drives, floppy, hard, disk, ide
"space" en comp.graphics, comp.os.ms-windows.misc, comp.sys.ibm.pc.hardware, comp.sys.mac.hardware, comp.windows.x: 2936 documentos, la palabra aparece en 79
palabra similar  coseno  docs del subcorpus
            hue   0.359                   3
     contiguous   0.279                   5
        decodes   0.251                   3
           ames   0.251                   7
        distant   0.247                   3

  Comparación: en el corpus completo -> nasa, shuttle, exploration, aeronautics, cfa


#### Interpretación: sentidos ocultos de *drive* y *space*

| Palabra | Corpus completo | Subcorpus |
|---|---|---|
| *drive* | drives, floppy, hard, disk, ide | **(vehículos)** talking, wheelie, shaft, address, drag |
| *space* | nasa, shuttle, exploration, aeronautics, cfa | **(computación)** hue, contiguous, decodes, ames, distant |

**El sentido mayoritario desaparece.** Al restringir la matriz a otro subconjunto de documentos, las palabras similares del corpus completo desaparecen por completo: ninguna relacionada con discos para *drive*, y casi ninguna astronómica para *space*. Esto confirma que el vector de una palabra refleja los contextos de los que se aprende, y que en el corpus completo el uso mayoritario oculta los demás.

**Emergen sentidos distintos de los previstos.** Esperábamos el sentido de "conducir" para *drive* y el de "espacio en disco" para *space*. Los resultados muestran algo más rico:
- *drive* en vehículos se asocia con *shaft*, *wheelie* y *drag*. *shaft* corresponde al sentido mecánico de transmisión: en motocicletas, la transmisión por cardán (*shaft drive*) frente a la de cadena. *wheelie* y *drag* (*drag racing*) pertenecen al vocabulario de manejo deportivo de motos.
- *space* en computación muestra dos sentidos técnicos: *contiguous* ("contiguous space", espacio contiguo en memoria o disco), que es el que buscábamos, y *hue*, asociado a "color space" (espacios de color como HSV o RGB en gráficos). Además, el sentido astronómico persiste incluso dentro de computación (*ames*, por NASA Ames, y *distant*), probablemente por publicaciones de comp.graphics sobre imágenes espaciales.

**Por qué no surge el sentido de "conducir".** La co-ocurrencia a nivel documento solo puede aislar los sentidos que tienen un vocabulario temático propio: unidad de disco, transmisión, espacio de color. El verbo "conducir" no tiene esa firma: aparece en cualquier publicación sobre vehículos, junto a cualquier palabra, y su vector no se distingue del vocabulario general del subcorpus. Capturar ese tipo de sentido requiere un contexto más corto, como la ventana de pocas palabras que usan los word embeddings, donde "drive" aparecería junto a palabras como *to*, *my* o *home*.

**Limitación: los subcorpus pequeños vuelven a introducir ruido.** Dentro de cada subcorpus, la palabra elegida aparece en solo 81 (*drive*) y 79 (*space*) documentos, y varias de sus similares en apenas 3 a 7. Con tan pocos documentos, uno o dos textos extensos pueden determinar la similitud, lo que explica asociaciones sin relación aparente como *talking*, *address* o *decodes*. Es el mismo problema de los términos raros observado antes, ahora causado por el tamaño reducido del corpus.

**Conclusión.** El experimento confirma que una palabra ambigua tiene varios sentidos presentes en los datos, y que el vector único del corpus completo solo refleja el mayoritario. También muestra los límites de la co-ocurrencia por documento: distingue sentidos temáticos, pero no usos generales, y necesita muchos documentos para producir similitudes confiables. Ambas limitaciones motivan las representaciones que se ven en la clase 2: los embeddings, que aprenden de contextos más cortos y de corpus mucho mayores, y, más adelante, los modelos contextuales, que asignan un vector distinto a cada aparición de una palabra.

## Conclusiones generales

| Modelo | F1 macro (test) |
|---|---|
| Azar | 0,050 |
| Prototipos (TF-IDF sin stop words) | 0,533 |
| Naïve Bayes de la plantilla | 0,585 |
| **ComplementNB optimizado** | **0,699** |

1. **TF-IDF captura información temática real.** Funciona cuando un tema tiene vocabulario propio (*pov*, *nhl*, *processor*). Un clasificador sin entrenamiento, basado solo en el documento más similar, supera al azar diez veces.

2. **Mide coincidencia de palabras, no de significado.** Los errores se repiten en los cuatro ejercicios con el mismo origen: homónimos (*peter*, *sound*, *processor*), palabras funcionales y términos raros que generan similitudes por coincidencia.

3. **El mejor ajuste depende del método.** Eliminar stop words fue clave para la similitud coseno (Ejercicios 1, 2 y 4), pero irrelevante para Naïve Bayes. Los términos raros perjudican la similitud entre palabras, pero ayudan a clasificar. En Naïve Bayes, el `alpha` explicó el 86% de la mejora.

4. **Agregar información supera a decidir con un solo ejemplo.** Naïve Bayes resume cada clase con cientos de documentos y diluye las coincidencias espurias. El prototipo depende de un único vecino.

5. **El límite de fondo es la polisemia.** Cada palabra tiene un solo vector y su sentido mayoritario oculta los demás (*drive*, *space*). Además, el documento completo como contexto captura temas, pero no usos generales. Esto motiva los word embeddings, que aprenden de ventanas cortas, y los modelos contextuales, que asignan a cada aparición de una palabra un vector distinto.